# Lab 9 Tasks: Raster Data

**Total: 100 points.** Work in this notebook. Answer every question in a markdown cell directly under the task. Submit the notebook with all outputs showing and an HTML export.

Use the same files as the walkthrough (`data/`). Every map needs a colorbar or a legend, and every statistic needs its unit and the cell size it came from.

In [1]:
import os, sys, importlib.util

# Some installs point GDAL at a missing or mismatched PROJ database ("Cannot find proj.db").
# Point it at the database that ships with rasterio, or the environment's own, before rasterio loads.
for d in (os.path.join(os.path.dirname(importlib.util.find_spec("rasterio").origin), "proj_data"),
          os.path.join(sys.prefix, "share", "proj")):
    if os.path.exists(os.path.join(d, "proj.db")):
        os.environ["PROJ_DATA"] = os.environ["PROJ_LIB"] = d
        break

import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Patch

print("rioxarray", rioxarray.__version__, "| geopandas", gpd.__version__)

rioxarray 0.22.0 | geopandas 1.1.3


The following helper functions may or may not come in handy when completing these tasks!

In [2]:
SEQ = "Purples"
ELEV = LinearSegmentedColormap.from_list("elev", ["#F5F3FB", "#2563EB", "#1E1B2E"])
NDVI_CM = LinearSegmentedColormap.from_list("ndvi", ["#DB2777", "#F5F3FB", "#059669"])


def extent(da):
    """Plot extent (left, right, bottom, top) of a raster, in its own coordinates."""
    left, bottom, right, top = da.rio.bounds()
    return (left, right, bottom, top)


def show(da, ax, title, cmap=SEQ, label="", **kw):
    """Draw a 2-D raster with a colorbar."""
    im = ax.imshow(da.values, cmap=cmap, extent=extent(da), **kw)
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label=label)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    return im

In [3]:
from rasterio import features


def zonal_sum(grid, zones):
    """Sum the cells of a 2-D raster inside each polygon.

    grid:  2-D DataArray (NaN cells count as 0).
    zones: GeoDataFrame in the same CRS as the grid.
    Returns one total per row of `zones`, in order. A cell belongs to the polygon that contains its center.
    """
    ids = features.rasterize(((geom, i + 1) for i, geom in enumerate(zones.geometry)),
                             out_shape=grid.shape, transform=grid.rio.transform(), fill=0, dtype="int32")
    totals = np.bincount(ids.ravel(), weights=np.nan_to_num(grid.values).ravel(), minlength=len(zones) + 1)
    return totals[1:]

In [4]:
def slope_degrees(elev, cell):
    """Slope in degrees from a 2-D elevation grid in meters with square cells of `cell` meters."""
    dz_dy, dz_dx = np.gradient(elev.values, cell, cell)    # meters of rise per meter of run
    return np.degrees(np.arctan(np.hypot(dz_dx, dz_dy)))

Loading some data for you:

In [5]:
# ESA WorldCover 2021 class codes that occur in this window (0 means no data).
WC = {10: ("Tree cover", "#059669"), 20: ("Shrubland", "#84CC16"), 30: ("Grassland", "#FACC15"),
      40: ("Cropland", "#F59E0B"), 50: ("Built-up", "#DB2777"), 60: ("Bare or sparse", "#B8B0C8"),
      80: ("Permanent water", "#2563EB"), 90: ("Herbaceous wetland", "#7C3AED")}

In [6]:
pop = rioxarray.open_rasterio("data/dallas_pop_2020.tif", masked=True).squeeze(drop=True)
tracts = gpd.read_file("data/dallas_tracts_acs2024.gpkg").to_crs(pop.rio.crs)
county = tracts.dissolve()
print(pop.dims, pop.shape, pop.rio.crs.to_string())

('y', 'x') (556, 775) ESRI:54009


In [7]:
dem = rioxarray.open_rasterio("data/dallas_dem_glo30.tif", masked=True).squeeze(drop=True)
dem_utm = dem.rio.reproject("EPSG:32614")
dem_utm = dem_utm.where(dem_utm > 0)          # reprojection fills the corners with 0: mask them
cell_m = abs(dem_utm.rio.resolution()[0])      # cell size in meters

In [8]:
s2 = rioxarray.open_rasterio("data/dallas_s2_2025-09-17.tif").assign_coords(band=["blue", "green", "red", "nir"])
red, nir = s2.sel(band="red").astype("float32"), s2.sel(band="nir").astype("float32")
ndvi = (nir - red) / (nir + red)

The cells above load the helpers and the data used below: `pop` (residents per 100 m cell), `tracts` (in the grid's CRS), `dem` and `dem_utm` (elevation), `ndvi`, and `red`, `nir`.

## Task 1 (10 points). Read two rasters

Open `data/dallas_dem_glo30.tif` (elevation) and `data/dallas_landcover_2021.tif` (land cover). For each, report the number of rows and columns, the CRS, the cell size, and the data type. Then say which of the two is continuous and which is categorical, and name one statistic that makes sense for one and not the other.

In [9]:
# your code here

## Task 2 (15 points). Coarsen a count

Coarsen `pop` to 500 m cells (blocks of 5 by 5) two ways: with the sum and with the mean. Report the total residents in the 100 m grid and in each coarse grid. Which method is correct for residents per cell, and why? Name one raster in this lab where the other method is the right choice.

In [11]:
# your code here

## Task 3 (15 points). NDVI

Using `ndvi` (loaded above, downtown window, cell size 10 m), make a map with a colorbar. Report the share of cells with NDVI above 0.3. In two or three sentences describe where the low-NDVI and high-NDVI cells fall and what is on the ground there. A true-color view helps: stack the `red`, `green`, and `blue` bands of `s2` and divide by 3,000 so the values fall between 0 and 1.

In [13]:
# your code here

## Task 4 (10 point). Smoothing and slope

Compute the slope of `dem_utm` with `slope_degrees(dem_utm, cell_m)` and report its 99th percentile. Then smooth the elevation with a 5 by 5 window and again with a 15 by 15 window (`rolling(x=n, y=n, center=True).mean()`), compute the slope of each, and report the 99th percentile. Explain the trend in one or two sentences.

In [15]:
# your code here

## Task 5 (20 points). Tract totals from the grid

Use `zonal_sum` to compute the total residents of every tract from `pop`. Report the grid total and the ACS total (`pop_total`) over all tracts, and the correlation between the two columns. Make a scatter plot with the line of equal values and a legend. List the three tracts (by `GEOID`) where the two sources differ most, and give one possible reason for the disagreement in one of them. State clearly that it is a hypothesis you did not test.

In [17]:
# your code here

## Task 6 (15 points). Cell size and density

Using the method from Part 8, compute the share of residents living in cells above 2,500 residents per square km for three cell sizes: 100 m, 500 m, and 2 km. Describe the trend. Then answer: if you were choosing where to place a clinic and wanted to find the densest neighborhoods, which of the three cell sizes would you use, and what would you lose by using the largest one?

In [19]:
# your code here

## Task 7 (15 points). Raster or vector? (open-ended)

For each question below, say whether you would store the data as a raster or as vector features, and give one reason. If you choose a raster, name a cell size and say what it can and cannot show. Then say what is lost in the conversion if the data started in the other form.

1. Which streets in a neighborhood will flood in a heavy storm?
2. What share of a city is covered by tree canopy, and where is canopy missing?

There is no single correct answer. Your reasoning is graded.

*Your answer here*